# 10 · Statistics and intervals reported in the paper
No training. Reproduces every statistical result in the manuscript and writes it to `results/paper/statistics/`.

1. H1 protocol audit (descriptive across models) and the sensitivity arms (S1)
2. Seed-level comparisons with XGBoost: seeds won, seed-level interval, exact Wilcoxon and Mann–Whitney with Holm
   correction, TOST at ±0.005 and ±0.02 (H2, also for S-C raw and S-C label-free), minority-class recall, H4
3. Friedman tests and Nemenyi critical difference per dataset and task; Kendall's τ across datasets (B6); H3 summary
4. Intervals for seed means from the two-level bootstrap: blocks of 200 consecutive test flows within each class
   (main), single flows within each class, and blocks of 50 and 1,000 (sensitivity)
5. Host structure of the test sets (why host-disjoint splits and host-cluster resampling are not used)

Section 4 takes roughly 1–2 hours on a laptop CPU. Set `RUN_HOST = True` to also compute the host-cluster
intervals discussed in the paper.

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 220, 'display.max_columns', 60, 'display.max_rows', 300)
from sdnids_bench.config import load_config, mode_params, active_datasets
cfg = load_config('..')
mp = mode_params(cfg)
print('mode:', cfg['mode'], '| results:', cfg['out_dir'])
from sdnids_bench.experiments import load_results, choose_best_L, ds_dir
from sdnids_bench import analysis as A, stats as S, figures as F, tables as TB, paper_figures as PF
DATASETS = active_datasets(cfg, 'B1')
PRIMARY = cfg['primary_dataset']
L = choose_best_L(cfg)                      # 8 in the paper (selected on validation data in B3)
PAPER = os.path.join(cfg['out_dir'], 'paper')
res = load_results(cfg, list(cfg['datasets']))
print('best L =', L); print(res.groupby(['dataset', 'exp']).size())
OUT = os.path.join(PAPER, 'statistics'); os.makedirs(OUT, exist_ok=True)
RUN_HOST = False
def out(df, name):
    df.to_csv(os.path.join(OUT, name + '.csv'), index=False); return df

## 1. Protocol audit (H1) and sensitivity arms

In [ ]:
pg = out(A.protocol_gaps(res), 'H1_protocol_gaps')
display(pg.round(3))
pm = out(pd.concat([A.per_model_gaps(res, ds, t, LL).assign(dataset=ds, task=t, L=LL)
                    for ds in DATASETS for t in cfg['tasks'] for LL in sorted({1, L})]), 'H1_per_model_gaps')
sens = A.sensitivity_table(res, L)
sens.to_csv(os.path.join(OUT, 'S1_sensitivity_macro_f1.csv'))
display(sens.round(4))
if 'SC_raw' in sens:
    d = (sens['SC_raw'] - sens['SC']).groupby(level=0)
    print('S-C raw minus S-C, mean [min, max] per dataset:'); display(pd.DataFrame({'mean': d.mean(), 'min': d.min(), 'max': d.max()}).round(4))

## 2. Comparisons with XGBoost (H2), minority recall, and H4

In [ ]:
parts = []
for ds in DATASETS:
    for t in cfg['tasks']:
        parts.append(A.h2_seed_stats(res, ds, t, L))
    for prot in ('SC_raw', 'SC_feat'):
        parts.append(A.h2_seed_stats(res, ds, 'multi', L, exp='S1', protocol=prot))
h2 = out(pd.concat([p for p in parts if len(p)], ignore_index=True), 'H2_seed_level')
display(h2.round(4))
h2m = out(pd.concat([A.h2_seed_stats(res, ds, 'multi', L, metric='minority_recall') for ds in DATASETS], ignore_index=True),
          'H2_minority_recall_seed_level')
display(h2m.round(4))
h4, h4cost = A.h4_gru_vs_transformer(res, PRIMARY, L)
out(h4, 'H4_gru_vs_transformer'); h4cost.to_csv(os.path.join(OUT, 'H4_cost.csv'))
display(h4.round(4)); display(h4cost)

## 3. Ranks (B2), agreement across datasets (B6) and temporal context (H3)

In [ ]:
fr = out(A.friedman_per_setting(res, DATASETS, L), 'friedman_per_setting'); display(fr)
M, ra = A.rank_agreement(res, DATASETS, {ds: L for ds in DATASETS})
out(ra, 'B6_rank_agreement'); display(ra)
h3, h3s = A.h3_summary(res, PRIMARY)
out(h3, 'H3_summary'); json.dump(h3s, open(os.path.join(OUT, 'H3_summary_over_models.json'), 'w'), indent=2)
display(h3.round(4)); print(h3s)

## 4. Two-level bootstrap intervals for seed means

In [ ]:
import time
t0 = time.time()
designs = ('block', 'strat') + (('host',) if RUN_HOST else ())
I = A.design_tables(res, cfg['out_dir'], cfg, DATASETS, designs=designs, block=200, B=1000, L=L, primary=PRIMARY,
                    log=lambda s: print(s, f'({(time.time() - t0) / 60:.1f} min)'))
for k, v in I.items():
    out(v, f'intervals_{k}')
    print(k); display(v.round(4))

In [ ]:
# block-length check for the main comparison (multi-class, S-C)
meta, smp, rows = A.TestMeta(cfg), {}, []
for ds in DATASETS:
    for blk in (50, 1000):
        rx, C = A._runs(res, cfg['out_dir'], meta, ds, 'multi', L, 'B1', 'SC', 'XGBoost', 'block', blk, smp)
        for m in sorted(res[(res.exp == 'B1') & (res.dataset == ds)].model.unique()):
            if m == 'XGBoost':
                continue
            rm, _ = A._runs(res, cfg['out_dir'], meta, ds, 'multi', L, 'B1', 'SC', m, 'block', blk, smp)
            lo, hi = S.design_paired_diff_ci([a for a, _ in rm], [a for a, _ in rx], rm[0][1], C, B=1000)
            rows.append({'dataset': ds, 'block': blk, 'model': m, 'ci_low': lo, 'ci_high': hi})
bs = out(pd.DataFrame(rows), 'intervals_h2_block_length_check'); display(bs.round(4))
print(f'done in {(time.time() - t0) / 60:.1f} min')

## 5. Host structure of the test sets

In [ ]:
hosts, per_class = A.host_diagnostics(cfg, DATASETS, L)
out(hosts, 'host_structure_test'); out(per_class, 'hosts_per_class')
display(hosts.round(3)); display(per_class.round(3))

## 6. Appendix tables B1–B3

In [ ]:
blk = I['h2'][I['h2'].design == 'block'].set_index(['dataset', 'task', 'protocol', 'model'])
b1 = h2.copy()
b1['block_ci_low'] = [blk.loc[(r.dataset, r.task, r.protocol, r.model), 'ci_low'] for _, r in b1.iterrows()]
b1['block_ci_high'] = [blk.loc[(r.dataset, r.task, r.protocol, r.model), 'ci_high'] for _, r in b1.iterrows()]
out(b1[['dataset', 'task', 'protocol', 'model', 'diff', 'seeds_won', 'seed_ci_low', 'seed_ci_high', 'block_ci_low',
        'block_ci_high', 'p_holm']], 'table_B1_comparisons_with_xgboost')
h3b = I['h3'][I['h3'].design == 'block'][['model', 'ci_low', 'ci_high']]
t = h3[h3.neural].assign(gain=lambda d: d.minority_recall_bestL - d.minority_recall_L1)
out(t[['model', 'best_L_val', 'gain']].merge(h3b, on='model').sort_values('gain', ascending=False), 'table_B2_h3_gains')
d = I['h2'][(I['h2'].task == 'multi') & (I['h2'].protocol == 'SC')]
d = pd.concat([d, bs.assign(design=lambda x: 'block' + x['block'].astype(str), task='multi', protocol='SC').drop(columns='block')])
b3 = d.pivot_table(index=['dataset', 'model'], columns='design', values=['ci_low', 'ci_high'])
b3.to_csv(os.path.join(OUT, 'table_B3_resampling_sensitivity.csv'))
display(b3.round(3))
print(sorted(os.listdir(OUT)))